# Buddy – fine-tune Whisper on your voice

Companion to `guides/01-11-custom-stt.md` (step 6). Runtime → Change runtime type → **T4 GPU**, then run the cells in order.

Input: `voice-data.tar.gz` from the Pi (step 5). Output: `ggml-buddy-base.en.bin` → `~/buddy/models/` (step 7).

## Cell 1 – setup

In [ ]:
!pip -q install jiwer soundfile
import copy, csv, random, re
import jiwer, soundfile as sf, torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

BASE = "openai/whisper-base.en"  # same size as BUDDY_STT_MODEL
dev = "cuda"

## Cell 2 – upload

In [ ]:
from google.colab import files
files.upload()
!tar xzf voice-data.tar.gz

## Cell 3 – load the clips

In [ ]:
proc = WhisperProcessor.from_pretrained(BASE)
rows = list(csv.DictReader(open("voice-data/metadata.csv")))
for r in rows:
    audio, sr = sf.read("voice-data/" + r["file_name"], dtype="float32")
    assert sr == 16000, r["file_name"]
    r["x"] = proc.feature_extractor(audio, sampling_rate=16000, return_tensors="pt").input_features[0]
random.seed(0)
random.shuffle(rows)
n_val = max(10, len(rows) // 7)
val, train = rows[:n_val], rows[n_val:]
print(len(train), "train,", len(val), "validation")

## Cell 4 – score before training

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(BASE).to(dev)

def norm(t):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", t.lower())).strip()

def score(rows):
    model.eval()
    heard = []
    with torch.no_grad():
        for i in range(0, len(rows), 16):
            x = torch.stack([r["x"] for r in rows[i:i + 16]]).to(dev)
            heard += proc.batch_decode(model.generate(x, max_new_tokens=64), skip_special_tokens=True)
    return jiwer.wer([norm(r["transcription"]) for r in rows], [norm(h) for h in heard]), heard

before, heard = score(val)
print(f"before: {before:.1%} word error rate")
for r, h in zip(val, heard):
    if norm(r["transcription"]) != norm(h):
        print(" ", r["transcription"], "→", h)

## Cell 5 – train

In [ ]:
def labels(rows):
    ids = [proc.tokenizer(r["transcription"]).input_ids for r in rows]
    y = torch.full((len(ids), max(map(len, ids))), -100)  # -100 = padding, ignored by the loss
    for i, t in enumerate(ids):
        y[i, :len(t)] = torch.tensor(t)
    if (y[:, 0] == model.config.decoder_start_token_id).all():
        y = y[:, 1:]  # the model puts the start token in front itself
    return y

model.config.apply_spec_augment = True  # randomly masks bits of the spectrogram: less overfitting ⚠️
opt = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
EPOCHS, BATCH = 5, 8
best = (before, copy.deepcopy(model.state_dict()))
for epoch in range(EPOCHS):
    model.train()
    random.shuffle(train)
    for i in range(0, len(train), BATCH):
        b = train[i:i + BATCH]
        x = torch.stack([r["x"] for r in b]).to(dev)
        loss = model(input_features=x, labels=labels(b).to(dev)).loss
        loss.backward()
        opt.step()
        opt.zero_grad()
    wer, _ = score(val)
    print(f"epoch {epoch + 1}: loss {loss.item():.3f}, word error rate {wer:.1%}")
    if wer < best[0]:
        best = (wer, copy.deepcopy(model.state_dict()))
model.load_state_dict(best[1])
print(f"kept: {best[0]:.1%} (before {before:.1%})")

## Cell 6 – convert to GGML and download

In [ ]:
model.save_pretrained("buddy-whisper")
proc.save_pretrained("buddy-whisper")
!git clone -q --depth 1 https://github.com/openai/whisper
!git clone -q --depth 1 https://github.com/ggml-org/whisper.cpp
!python whisper.cpp/models/convert-h5-to-ggml.py buddy-whisper whisper .
!mv ggml-model.bin ggml-buddy-base.en.bin
files.download("ggml-buddy-base.en.bin")